# 🧠 Universal AI Document Analyzer
## Kaggle Experimentation & Artifact Generation Notebook

**Academic Deep Learning Project**

---

### What this notebook does:

| Stage | Description |
|---|---|
| 1 | Dataset discovery & statistics |
| 2 | PDF/image detection & PDF-to-image conversion |
| 3 | OCR using **EasyOCR** (pretrained model) |
| 4 | Text cleaning & normalization |
| 5 | Text chunking (300 words, 50 overlap) |
| 6 | Embedding generation using **all-MiniLM-L6-v2** |
| 7 | FAISS vector index creation |
| 8 | Retrieval testing & evaluation |
| 9 | Visualization |
| 10 | Artifact export |

> **NOTE:** We use the **pretrained** `all-MiniLM-L6-v2` Transformer model to generate
> semantic embeddings for document chunks and queries. We do NOT train this model from scratch.

---

### Dataset Upload Instructions:

1. On Kaggle: Click **Add Data → Upload Dataset**
2. Upload a ZIP file containing your documents with this structure:
```
dataset/
  resumes/
    file1.pdf
    file2.jpg
    ...
```
3. Or place files directly in the dataset root
4. Supported: `.pdf`, `.jpg`, `.jpeg`, `.png`

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
# Run this cell first (Kaggle already has most packages)

!pip install -q easyocr sentence-transformers faiss-cpu pdf2image PyPDF2 pillow

# Install Poppler for PDF-to-image conversion (Linux/Kaggle)
!apt-get install -qq poppler-utils 2>/dev/null || echo 'Poppler already installed or not needed'

print('✅ Dependencies installed')

In [1]:
# ============================================================
# CELL 2: Imports & Configuration
# ============================================================

import os
import re
import glob
import json
import pickle
import warnings
import time
from pathlib import Path
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from PIL import Image

warnings.filterwarnings('ignore')

# ── Configuration (all configurable here) ──────────────────
CONFIG = {
    # Dataset path — Kaggle mounts datasets under /kaggle/input/
    'DATASET_ROOT':       '/kaggle/input',
    
    # Supported file extensions
    'SUPPORTED_EXTS':     ['.pdf', '.jpg', '.jpeg', '.png'],
    
    # OCR
    'OCR_LANGUAGES':      ['en'],
    'OCR_GPU':            False,    # Set True if Kaggle GPU is enabled
    
    # PDF conversion
    'PDF_DPI':            200,
    
    # Text chunking
    'CHUNK_SIZE':         300,      # words per chunk
    'CHUNK_OVERLAP':      50,       # overlapping words
    
    # Embedding model (pretrained — NOT trained from scratch)
    'EMBEDDING_MODEL':    'sentence-transformers/all-MiniLM-L6-v2',
    'EMBEDDING_DIM':      384,
    
    # Retrieval
    'TOP_K':              5,
    'SIMILARITY_THRESHOLD': 0.3,
    
    # Artifacts output path
    'ARTIFACTS_DIR':      '/kaggle/working/artifacts',
}

# Create artifact directories
os.makedirs(f"{CONFIG['ARTIFACTS_DIR']}/faiss",    exist_ok=True)
os.makedirs(f"{CONFIG['ARTIFACTS_DIR']}/metadata", exist_ok=True)

print('Configuration loaded:')
for k, v in CONFIG.items():
    print(f'  {k:25s} = {v}')

Configuration loaded:
  DATASET_ROOT              = /kaggle/input
  SUPPORTED_EXTS            = ['.pdf', '.jpg', '.jpeg', '.png']
  OCR_LANGUAGES             = ['en']
  OCR_GPU                   = False
  PDF_DPI                   = 200
  CHUNK_SIZE                = 300
  CHUNK_OVERLAP             = 50
  EMBEDDING_MODEL           = sentence-transformers/all-MiniLM-L6-v2
  EMBEDDING_DIM             = 384
  TOP_K                     = 5
  SIMILARITY_THRESHOLD      = 0.3
  ARTIFACTS_DIR             = /kaggle/working/artifacts


In [2]:
# ============================================================
# CELL 3: Stage 1 — Dataset Discovery
# ============================================================

def discover_documents(root_path: str, supported_exts: List[str]) -> List[Dict]:
    """
    Recursively discover all supported documents under root_path.
    Returns a list of document info dicts.
    """
    documents = []
    root = Path(root_path)
    
    if not root.exists():
        print(f'WARNING: Dataset root not found: {root_path}')
        return documents
    
    for ext in supported_exts:
        # Search recursively
        for filepath in root.rglob(f'*{ext}'):
            documents.append({
                'document_id':  str(hash(str(filepath)))[:12],
                'filename':     filepath.name,
                'filepath':     str(filepath),
                'extension':    ext.lower(),
                'file_type':    'pdf' if ext == '.pdf' else 'image',
                'file_size_kb': round(filepath.stat().st_size / 1024, 2),
            })
    
    return documents


# Discover dataset
all_docs = discover_documents(CONFIG['DATASET_ROOT'], CONFIG['SUPPORTED_EXTS'])

# ── Summary statistics ─────────────────────────────────────
total_files  = len(all_docs)
pdf_count    = sum(1 for d in all_docs if d['extension'] == '.pdf')
image_count  = total_files - pdf_count
total_size   = sum(d['file_size_kb'] for d in all_docs)

print('=' * 50)
print('📂 DATASET DISCOVERY RESULTS')
print('=' * 50)
print(f'  Total files found  : {total_files}')
print(f'  PDF files          : {pdf_count}')
print(f'  Image files        : {image_count}')
print(f'  Total size         : {total_size:.1f} KB ({total_size/1024:.2f} MB)')
print()

if total_files == 0:
    print('⚠️  No documents found! Please upload your dataset.')
    print(f'   Expected location: {CONFIG["DATASET_ROOT"]}')
else:
    print('Sample files discovered:')
    for doc in all_docs[:5]:
        print(f'  [{doc["extension"]}] {doc["filename"]} ({doc["file_size_kb"]} KB)')
    if total_files > 5:
        print(f'  ... and {total_files - 5} more')

# ── Visualization: File type distribution ─────────────────
if total_files > 0:
    ext_counts = {}
    for doc in all_docs:
        ext_counts[doc['extension']] = ext_counts.get(doc['extension'], 0) + 1
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Dataset Overview', fontsize=14, fontweight='bold', color='#333')
    
    # Pie chart
    axes[0].pie(
        ext_counts.values(),
        labels=[e.upper() for e in ext_counts.keys()],
        autopct='%1.1f%%',
        colors=['#6c63ff', '#a78bfa', '#22d3a4', '#60a5fa'],
        startangle=90,
    )
    axes[0].set_title('File Type Distribution')
    
    # Bar chart
    sizes = [d['file_size_kb'] for d in all_docs]
    axes[1].hist(sizes, bins=min(20, total_files), color='#6c63ff', edgecolor='white', alpha=0.85)
    axes[1].set_xlabel('File Size (KB)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('File Size Distribution')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['ARTIFACTS_DIR']}/dataset_overview.png", dpi=150, bbox_inches='tight')
    plt.show()
    print('📊 Chart saved.')

📂 DATASET DISCOVERY RESULTS
  Total files found  : 0
  PDF files          : 0
  Image files        : 0
  Total size         : 0.0 KB (0.00 MB)

⚠️  No documents found! Please upload your dataset.
   Expected location: /kaggle/input


In [ ]:
# ============================================================
# CELL 4: Stage 2 — Text Cleaning Functions
# ============================================================

def clean_text(raw_text: str) -> str:
    """
    Clean OCR-extracted text while preserving document semantics.
    Conservative cleaning — do not remove meaningful content.
    """
    if not raw_text or not raw_text.strip():
        return ''
    
    text = raw_text
    
    # Unicode normalization
    text = text.replace('\u2019', "'").replace('\u2018', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = text.replace('\u2013', '-').replace('\u2014', '-')
    text = text.replace('\u00a0', ' ').replace('\u200b', '').replace('\ufeff', '')
    
    # Remove control characters
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    
    # Normalize line endings
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    
    # Collapse excessive newlines
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    # Remove excessive spaces
    text = re.sub(r'[ \t]{3,}', ' ', text)
    
    # Clean line by line
    cleaned_lines = []
    for line in text.split('\n'):
        stripped = line.strip()
        if stripped == '':
            cleaned_lines.append('')
            continue
        if re.fullmatch(r'[-=_*#|~.+]{3,}', stripped):
            continue  # Remove divider lines
        cleaned_lines.append(stripped)
    
    text = '\n'.join(cleaned_lines)
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text.strip()


def chunk_text(text: str, document_id: str, filename: str,
               page: int = None, chunk_size: int = 300, overlap: int = 50) -> List[Dict]:
    """
    Split text into overlapping word-based chunks with metadata.
    """
    words = text.split()
    if not words:
        return []
    
    chunks = []
    chunk_id = 0
    start = 0
    
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk_text_str = ' '.join(words[start:end])
        chunks.append({
            'document_id': document_id,
            'filename':    filename,
            'page':        page,
            'chunk_id':    chunk_id,
            'text':        chunk_text_str,
            'word_count':  len(words[start:end]),
        })
        chunk_id += 1
        if end == len(words):
            break
        start += chunk_size - overlap
    
    return chunks


print('✅ Text cleaning and chunking functions defined')

In [ ]:
# ============================================================
# CELL 5: Stage 3 — Initialize EasyOCR
# ============================================================
# NOTE: We use the PRETRAINED EasyOCR model.
# No training is performed here.

import easyocr

print('Initializing EasyOCR (pretrained model)...')
print('This downloads the model on first run.')
print()

ocr_reader = easyocr.Reader(
    CONFIG['OCR_LANGUAGES'],
    gpu=CONFIG['OCR_GPU'],
)

print('✅ EasyOCR initialized')
print(f'   Languages: {CONFIG["OCR_LANGUAGES"]}')
print(f'   GPU mode : {CONFIG["OCR_GPU"]}')

In [ ]:
# ============================================================
# CELL 6: Stage 4 — OCR Processing Pipeline
# ============================================================

from pdf2image import convert_from_path

def ocr_image(pil_image) -> str:
    """Run EasyOCR on a PIL Image. Returns extracted text string."""
    img_array = np.array(pil_image.convert('RGB'))
    results = ocr_reader.readtext(img_array, detail=0, paragraph=True)
    return ' '.join(results)


def process_pdf(filepath: str) -> Dict:
    """Process a PDF — try direct extraction first, fall back to OCR."""
    try:
        import PyPDF2
        page_texts = []
        with open(filepath, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            pdf_images = None
            try:
                pdf_images = convert_from_path(filepath, dpi=CONFIG['PDF_DPI'])
            except Exception:
                pass
            
            for page_num, page in enumerate(reader.pages, start=1):
                direct_text = (page.extract_text() or '').strip()
                if len(direct_text) > 50:
                    text = direct_text
                elif pdf_images and page_num <= len(pdf_images):
                    text = ocr_image(pdf_images[page_num - 1])
                else:
                    text = ''
                page_texts.append({'page': page_num, 'text': clean_text(text)})
        
        combined = '\n\n'.join(p['text'] for p in page_texts if p['text'])
        return {'success': True, 'pages': len(page_texts), 'page_texts': page_texts,
                'text': combined, 'error': None}
    
    except Exception as e:
        return {'success': False, 'pages': 0, 'page_texts': [], 'text': '', 'error': str(e)}


def process_image_file(filepath: str) -> Dict:
    """Process a single image file."""
    try:
        img = Image.open(filepath).convert('RGB')
        raw_text = ocr_image(img)
        text = clean_text(raw_text)
        return {'success': True, 'pages': 1,
                'page_texts': [{'page': 1, 'text': text}],
                'text': text, 'error': None}
    except Exception as e:
        return {'success': False, 'pages': 0, 'page_texts': [], 'text': '', 'error': str(e)}


# ── Process all documents ─────────────────────────────────────
print('=' * 50)
print('🔍 STARTING OCR PROCESSING PIPELINE')
print('=' * 50)

ocr_results = []
failed_docs  = []
processed    = 0

for i, doc in enumerate(all_docs):
    print(f'[{i+1}/{len(all_docs)}] Processing: {doc["filename"]}', end=' ... ')
    t0 = time.time()
    
    if doc['extension'] == '.pdf':
        result = process_pdf(doc['filepath'])
    else:
        result = process_image_file(doc['filepath'])
    
    elapsed = time.time() - t0
    
    if result['success'] and result['text']:
        doc.update(result)
        ocr_results.append(doc)
        processed += 1
        char_count = len(result['text'])
        print(f'✅  {result["pages"]} page(s), {char_count} chars ({elapsed:.1f}s)')
    else:
        failed_docs.append({'filename': doc['filename'], 'error': result.get('error', 'Unknown')})
        print(f'❌  FAILED: {result.get("error", "Unknown error")}')

print()
print('=' * 50)
print(f'PROCESSING SUMMARY')
print(f'  Total files      : {len(all_docs)}')
print(f'  Successfully processed : {processed}')
print(f'  Failed           : {len(failed_docs)}')
print('=' * 50)

if failed_docs:
    print()
    print('Failed documents:')
    for fd in failed_docs:
        print(f'  ❌ {fd["filename"]}: {fd["error"]}')

In [ ]:
# ============================================================
# CELL 7: Stage 5 — Sample OCR Output
# ============================================================

if ocr_results:
    sample = ocr_results[0]
    print('=' * 60)
    print(f'📄 SAMPLE OCR OUTPUT')
    print(f'   File: {sample["filename"]}')
    print(f'   Pages: {sample["pages"]}')
    print(f'   Total chars: {len(sample["text"])}')
    print('=' * 60)
    print()
    print('--- RAW EXTRACTED TEXT (first 1000 chars) ---')
    print(sample['text'][:1000])
    print()
    print('...' if len(sample['text']) > 1000 else '')
else:
    print('No documents were successfully processed.')

In [ ]:
# ============================================================
# CELL 8: Stage 6 — Text Chunking
# ============================================================

all_chunks = []
global_chunk_id = 0

for doc in ocr_results:
    for page_info in doc.get('page_texts', []):
        page_chunks = chunk_text(
            text=page_info['text'],
            document_id=doc['document_id'],
            filename=doc['filename'],
            page=page_info['page'],
            chunk_size=CONFIG['CHUNK_SIZE'],
            overlap=CONFIG['CHUNK_OVERLAP'],
        )
        for chunk in page_chunks:
            chunk['chunk_id'] = global_chunk_id
            global_chunk_id += 1
            all_chunks.append(chunk)

# ── Statistics ──────────────────────────────────────────────
print('=' * 50)
print('✂️  CHUNKING RESULTS')
print('=' * 50)
print(f'  Chunk size          : {CONFIG["CHUNK_SIZE"]} words')
print(f'  Chunk overlap       : {CONFIG["CHUNK_OVERLAP"]} words')
print(f'  Total chunks created: {len(all_chunks)}')
print()

if all_chunks:
    word_counts = [c['word_count'] for c in all_chunks]
    print(f'  Avg words/chunk     : {np.mean(word_counts):.1f}')
    print(f'  Min words/chunk     : {min(word_counts)}')
    print(f'  Max words/chunk     : {max(word_counts)}')
    print()
    print('SAMPLE CHUNK:')
    sample_chunk = all_chunks[min(2, len(all_chunks)-1)]
    print(json.dumps({
        'document_id': sample_chunk['document_id'],
        'filename':    sample_chunk['filename'],
        'chunk_id':    sample_chunk['chunk_id'],
        'page':        sample_chunk['page'],
        'word_count':  sample_chunk['word_count'],
        'text':        sample_chunk['text'][:200] + '...',
    }, indent=2))

# ── Chunk word count histogram ──────────────────────────────
if len(all_chunks) > 1:
    plt.figure(figsize=(8, 4))
    plt.hist(word_counts, bins=20, color='#6c63ff', edgecolor='white', alpha=0.85)
    plt.xlabel('Words per Chunk')
    plt.ylabel('Number of Chunks')
    plt.title(f'Chunk Word Count Distribution (n={len(all_chunks)})')
    plt.axvline(np.mean(word_counts), color='#22d3a4', linestyle='--', label=f'Mean: {np.mean(word_counts):.0f}')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{CONFIG['ARTIFACTS_DIR']}/chunk_distribution.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================================
# CELL 9: Stage 7 — Embedding Generation
# ============================================================
#
# IMPORTANT NOTE ON TRAINING:
# We use the PRETRAINED sentence-transformers/all-MiniLM-L6-v2 model
# to generate semantic embeddings for document chunks and queries.
# We do NOT train this model from scratch.
# The pretrained model is downloaded from HuggingFace Hub.

from sentence_transformers import SentenceTransformer

print(f'Loading pretrained embedding model: {CONFIG["EMBEDDING_MODEL"]}')
print('(Downloads from HuggingFace Hub on first run)')
print()

embedding_model = SentenceTransformer(CONFIG['EMBEDDING_MODEL'])
actual_dim = embedding_model.get_sentence_embedding_dimension()

print(f'✅ Model loaded')
print(f'   Model: {CONFIG["EMBEDDING_MODEL"]}')
print(f'   Embedding dimension: {actual_dim}')
print()

if not all_chunks:
    print('⚠️  No chunks to embed. Please process documents first.')
else:
    print(f'Generating embeddings for {len(all_chunks)} chunks...')
    print('(This may take a few minutes)')
    
    chunk_texts = [c['text'] for c in all_chunks]
    
    embeddings = embedding_model.encode(
        chunk_texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # L2 normalize for cosine similarity
    )
    
    print()
    print('=' * 50)
    print('🧠 EMBEDDING RESULTS')
    print('=' * 50)
    print(f'  Number of chunks       : {len(all_chunks)}')
    print(f'  Embedding shape        : {embeddings.shape}')
    print(f'  Embedding dimension    : {embeddings.shape[1]}')
    print(f'  Data type              : {embeddings.dtype}')
    print()
    print(f'Example embedding (first 10 values of chunk 0):')
    print(f'  {embeddings[0, :10]}')

In [ ]:
# ============================================================
# CELL 10: Stage 8 — FAISS Vector Index
# ============================================================

import faiss

print('Building FAISS vector index...')
print(f'  Index type      : IndexFlatIP (inner product = cosine similarity)')
print(f'  Dimension       : {actual_dim}')
print(f'  Vectors to index: {len(all_chunks)}')
print()

# Create FAISS index
index = faiss.IndexFlatIP(actual_dim)

# Add embeddings (ensure float32)
embeddings_f32 = embeddings.astype(np.float32)
index.add(embeddings_f32)

print(f'✅ FAISS index built')
print(f'   Total vectors indexed: {index.ntotal}')
print()

# Save index
index_path    = f"{CONFIG['ARTIFACTS_DIR']}/faiss/document_index.faiss"
metadata_path = f"{CONFIG['ARTIFACTS_DIR']}/metadata/chunks_metadata.pkl"

faiss.write_index(index, index_path)
print(f'✅ FAISS index saved: {index_path}')

with open(metadata_path, 'wb') as f:
    pickle.dump(all_chunks, f)
print(f'✅ Metadata saved  : {metadata_path}')

In [ ]:
# ============================================================
# CELL 11: Stage 9 — Semantic Search Testing
# ============================================================

def semantic_search(query: str, top_k: int = 5) -> List[Dict]:
    """
    Embed the query and retrieve the top-k most similar chunks.
    """
    # Embed query
    q_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    
    # FAISS search
    distances, indices = index.search(q_embedding, top_k)
    
    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), start=1):
        if idx < 0 or idx >= len(all_chunks):
            continue
        chunk = all_chunks[idx]
        results.append({
            **chunk,
            'similarity': round(float(dist), 4),
            'rank':       rank,
        })
    return results


# ── Test queries ──────────────────────────────────────────────
test_queries = [
    'What are the technical skills and programming languages?',
    'What is the educational background and qualifications?',
    'What work experience and job roles are mentioned?',
]

print('=' * 60)
print('🔍 SEMANTIC SEARCH TEST RESULTS')
print('=' * 60)

for query in test_queries:
    print(f'\nQuery: "{query}"')
    print('-' * 50)
    results = semantic_search(query, top_k=CONFIG['TOP_K'])
    
    if not results:
        print('  No results found.')
    else:
        for r in results:
            print(f'  Rank {r["rank"]:>2} | Sim: {r["similarity"]:.4f} | '
                  f'{r["filename"]} | Page {r["page"]} | Chunk #{r["chunk_id"]}')
            print(f'         Text: {r["text"][:150]}...')

In [ ]:
# ============================================================
# CELL 12: Stage 10 — Retrieval Evaluation
# ============================================================
#
# NOTE: Since we do not have pre-labelled query→document pairs,
# we create a SMALL MANUALLY VERIFIED evaluation set below.
# This is clearly identified as manually created evaluation data.
# We report only actual retrieval results — no fabricated metrics.

def precision_at_k(retrieved_ids: List, relevant_ids: List, k: int) -> float:
    """P@K = number of relevant docs in top-K / K"""
    top_k = retrieved_ids[:k]
    return sum(1 for r in top_k if r in relevant_ids) / k if k > 0 else 0.0

def recall_at_k(retrieved_ids: List, relevant_ids: List, k: int) -> float:
    """R@K = number of relevant docs in top-K / total relevant"""
    if not relevant_ids:
        return 0.0
    top_k = retrieved_ids[:k]
    return sum(1 for r in top_k if r in relevant_ids) / len(relevant_ids)

def mrr(retrieved_ids_per_query: List[List], relevant_ids_per_query: List[List]) -> float:
    """Mean Reciprocal Rank"""
    rr_scores = []
    for retrieved, relevant in zip(retrieved_ids_per_query, relevant_ids_per_query):
        for rank, doc_id in enumerate(retrieved, start=1):
            if doc_id in relevant:
                rr_scores.append(1.0 / rank)
                break
        else:
            rr_scores.append(0.0)
    return np.mean(rr_scores) if rr_scores else 0.0


print('=' * 60)
print('📊 RETRIEVAL EVALUATION')
print('=' * 60)
print()

if len(all_chunks) < 2:
    print('⚠️  Not enough chunks for evaluation. Process more documents.')
else:
    # ── Manually created evaluation set ───────────────────────
    # These are sample queries matched to expected filenames.
    # Label format: {query_text: [list of expected filename substrings]}
    # Adjust these based on your actual dataset.
    print('⚠️  NOTE: Using manually created evaluation set.')
    print('   Queries and relevant documents are hand-labelled.')
    print()
    
    eval_queries = [
        {'query': 'Python programming skills',    'relevant_keywords': ['python', 'programming']},
        {'query': 'machine learning experience',  'relevant_keywords': ['machine learning', 'ml', 'deep learning']},
        {'query': 'university education degree',  'relevant_keywords': ['university', 'bachelor', 'degree', 'education']},
    ]
    
    K = CONFIG['TOP_K']
    precision_scores = []
    recall_scores    = []
    similarity_scores = []
    
    for eq in eval_queries:
        query    = eq['query']
        keywords = eq['relevant_keywords']
        
        results = semantic_search(query, top_k=K)
        
        # A chunk is "relevant" if its text contains any of the keywords
        relevant_chunk_ids = [
            c['chunk_id'] for c in all_chunks
            if any(kw.lower() in c['text'].lower() for kw in keywords)
        ]
        retrieved_chunk_ids = [r['chunk_id'] for r in results]
        
        p = precision_at_k(retrieved_chunk_ids, relevant_chunk_ids, K)
        r = recall_at_k(retrieved_chunk_ids, relevant_chunk_ids, K)
        avg_sim = np.mean([res['similarity'] for res in results]) if results else 0.0
        
        precision_scores.append(p)
        recall_scores.append(r)
        similarity_scores.append(avg_sim)
        
        print(f'Query : "{query}"')
        print(f'  P@{K}         = {p:.4f}')
        print(f'  R@{K}         = {r:.4f}')
        print(f'  Avg Similarity = {avg_sim:.4f}')
        print(f'  Relevant chunks found : {len(relevant_chunk_ids)}')
        print()
    
    mean_p   = np.mean(precision_scores)
    mean_r   = np.mean(recall_scores)
    mean_sim = np.mean(similarity_scores)
    
    print('-' * 50)
    print(f'MEAN Precision@{K} : {mean_p:.4f}')
    print(f'MEAN Recall@{K}    : {mean_r:.4f}')
    print(f'MEAN Avg Sim       : {mean_sim:.4f}')
    
    # ── Evaluation visualization ─────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    query_labels = [f'Q{i+1}' for i in range(len(eval_queries))]
    x = np.arange(len(query_labels))
    w = 0.35
    
    axes[0].bar(x - w/2, precision_scores, w, label=f'P@{K}', color='#6c63ff', alpha=0.85)
    axes[0].bar(x + w/2, recall_scores,    w, label=f'R@{K}', color='#22d3a4', alpha=0.85)
    axes[0].set_xlabel('Query')
    axes[0].set_ylabel('Score')
    axes[0].set_title(f'Precision@{K} and Recall@{K}')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(query_labels)
    axes[0].legend()
    axes[0].set_ylim(0, 1.1)
    axes[0].grid(axis='y', alpha=0.3)
    
    axes[1].bar(query_labels, similarity_scores, color='#a78bfa', alpha=0.85)
    axes[1].set_xlabel('Query')
    axes[1].set_ylabel('Avg Similarity Score')
    axes[1].set_title('Average Similarity per Query')
    axes[1].set_ylim(0, 1.0)
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.suptitle('Retrieval Evaluation Results', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{CONFIG['ARTIFACTS_DIR']}/retrieval_evaluation.png", dpi=150, bbox_inches='tight')
    plt.show()
    print('📊 Evaluation charts saved.')

In [ ]:
# ============================================================
# CELL 13: Stage 11 — Embedding Space Visualization (PCA)
# ============================================================

from sklearn.decomposition import PCA

if len(all_chunks) >= 5:
    print('Visualizing embedding space using PCA...')
    
    # Reduce to 2D
    pca = PCA(n_components=2, random_state=42)
    reduced = pca.fit_transform(embeddings[:min(500, len(embeddings))])
    
    # Color by document
    unique_docs = list({c['filename'] for c in all_chunks[:len(reduced)]})
    color_map = {doc: plt.cm.tab10(i / max(len(unique_docs), 1)) for i, doc in enumerate(unique_docs)}
    colors = [color_map[c['filename']] for c in all_chunks[:len(reduced)]]
    
    plt.figure(figsize=(10, 7))
    scatter = plt.scatter(reduced[:, 0], reduced[:, 1], c=colors, s=25, alpha=0.6, edgecolors='white', linewidths=0.3)
    
    # Legend
    patches = [mpatches.Patch(color=color_map[doc], label=doc[:30]) for doc in unique_docs]
    plt.legend(handles=patches, loc='upper right', fontsize=7, title='Documents')
    
    plt.xlabel(f'PCA Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    plt.ylabel(f'PCA Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    plt.title(f'Document Chunk Embeddings — PCA Projection (n={len(reduced)} chunks)')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{CONFIG['ARTIFACTS_DIR']}/embedding_pca.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ PCA visualization saved.')
    print(f'   Explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%')
else:
    print('Not enough chunks for PCA visualization.')

In [ ]:
# ============================================================
# CELL 14: Stage 12 — Save Artifacts & Config
# ============================================================

import json
from datetime import datetime

# Save artifact config
artifact_config = {
    'embedding_model':    CONFIG['EMBEDDING_MODEL'],
    'embedding_dimension': actual_dim,
    'chunk_size':         CONFIG['CHUNK_SIZE'],
    'chunk_overlap':      CONFIG['CHUNK_OVERLAP'],
    'top_k':              CONFIG['TOP_K'],
    'similarity_threshold': CONFIG['SIMILARITY_THRESHOLD'],
    'faiss_index_path':   'artifacts/faiss/document_index.faiss',
    'metadata_path':      'artifacts/metadata/chunks_metadata.pkl',
    'total_chunks':       len(all_chunks),
    'total_documents':    len(ocr_results),
    'created_at':         datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'version':            '1.0.0',
}

config_path = f"{CONFIG['ARTIFACTS_DIR']}/config.json"
with open(config_path, 'w') as f:
    json.dump(artifact_config, f, indent=2)

# Save chunk DataFrame
df_chunks = pd.DataFrame([{k: v for k, v in c.items() if k != 'text'} for c in all_chunks])
df_path = f"{CONFIG['ARTIFACTS_DIR']}/chunks_summary.csv"
df_chunks.to_csv(df_path, index=False)

print('=' * 60)
print('✅ ALL ARTIFACTS SAVED')
print('=' * 60)
print()
print('Files saved:')
for fpath in [
    f"{CONFIG['ARTIFACTS_DIR']}/faiss/document_index.faiss",
    f"{CONFIG['ARTIFACTS_DIR']}/metadata/chunks_metadata.pkl",
    config_path,
    df_path,
    f"{CONFIG['ARTIFACTS_DIR']}/dataset_overview.png",
    f"{CONFIG['ARTIFACTS_DIR']}/chunk_distribution.png",
    f"{CONFIG['ARTIFACTS_DIR']}/retrieval_evaluation.png",
    f"{CONFIG['ARTIFACTS_DIR']}/embedding_pca.png",
]:
    exists = os.path.exists(fpath)
    size   = os.path.getsize(fpath) / 1024 if exists else 0
    print(f'  {"✅" if exists else "❌"} {fpath} ({size:.1f} KB)')

print()
print('Artifact Config:')
print(json.dumps(artifact_config, indent=2))

In [ ]:
# ============================================================
# CELL 15: Final Summary — Review 1 Presentation Outputs
# ============================================================

print('=' * 65)
print('🎓 UNIVERSAL AI DOCUMENT ANALYZER — REVIEW 1 SUMMARY')
print('=' * 65)
print()
print(f'  1. Dataset Size          : {len(all_docs)} documents')
print(f'  2. PDF files             : {pdf_count}')
print(f'  3. Image files           : {image_count}')
print(f'  4. Successfully Processed: {processed}')
print(f'  5. Failed files          : {len(failed_docs)}')
print()
print(f'  6. Total Chunks Created  : {len(all_chunks)}')
print(f'     Chunk Size            : {CONFIG["CHUNK_SIZE"]} words')
print(f'     Chunk Overlap         : {CONFIG["CHUNK_OVERLAP"]} words')
print()
print(f'  7. Embedding Model       : {CONFIG["EMBEDDING_MODEL"]}')
print(f'     NOTE: Pretrained model. NOT trained from scratch.')

if 'embeddings' in dir() and embeddings is not None:
    print(f'  8. Embedding Shape       : {embeddings.shape}')
    print(f'  9. Embedding Dimension   : {embeddings.shape[1]}')

print()
print(f' 10. FAISS Index            : IndexFlatIP (exact inner product)')
print(f'     Vectors Indexed        : {index.ntotal}')
print()
print(f' 11. Artifacts Saved        : {CONFIG["ARTIFACTS_DIR"]}/')
print(f'     ├─ faiss/document_index.faiss')
print(f'     ├─ metadata/chunks_metadata.pkl')
print(f'     ├─ config.json')
print(f'     └─ chunks_summary.csv')
print()
print('Architecture:')
print('  PDF/Image → EasyOCR → Cleaning → Chunking')
print('  → all-MiniLM-L6-v2 Embeddings → FAISS Index')
print('  → Semantic Search → RAG → LLM Answer')
print()
print('✅ Notebook execution complete.')
print('   Download /kaggle/working/artifacts/ and place in project root.')